# Test Fine-tuning Results
Test the fine-tuned LIANet Creoss region performance to the local model perfomance

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from rasterio.windows import Window
from utils import s2_to_rgb, _preprocess_S2
# Add src to path
sys.path.insert(0, '/home/user/src')
import rasterio as rio
from datasets import PASTIS
from models.models_finetune import UNet, MicroUNet
from utils import _preprocess_S2
from settings import *
from datetime import datetime



from torchmetrics import MetricCollection

from metrics import multiclass_segmentation_metrics
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
import json
from models.models_finetune import DownstreamModel
from models.LIANet import LIANetLight
import omegaconf, hydra

pretrained_model_path = "/home/user/results_shared/fourier_learned_4regions/2026-03-18_19-06-01"


def load_model(CKPT_PATH, other_task):
        # # Get config from checkpoint directory
    # finetune_config_path = os.path.join(os.path.dirname(CKPT_PATH), "config.json")
    # if os.path.exists(finetune_config_path):
    #     with open(finetune_config_path, 'r') as f:
    #         config = json.load(f)
    #     model_type = config.get("model_type", "replace_final_block")


    # Create DownstreamModel instance - CR finetuning
    # print(f"Loading DownstreamModel with adaptation strategy '{model_type}'...")
    model_finetune = DownstreamModel(
        model_path=pretrained_model_path,
        checkpoint_path_relative="model_checkpoints/latest_validation_checkpoint.pt",
        adaption_strategy="replace_final_block",
        num_classes=num_classes[other_task],
        activation="none"
    )

    checkpoint = torch.load(CKPT_PATH, map_location=device)
    state_dict = checkpoint["model_state_dict"]
    if state_dict and all(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

    model_finetune.load_state_dict(state_dict, strict=True)
    model_finetune = model_finetune.to(device)
    model_finetune.eval()
    # print("✓ Model loaded successfully")
    return model_finetune

In [3]:
import os
import torch
import pandas as pd

from tqdm import tqdm
from torchmetrics import MetricCollection
from metrics import multiclass_segmentation_metrics


all_tasks_list = {
    "PASTIS_joint_T31TFM",
    "PASTIS_joint_T31TFJ",
    "PASTIS_joint_T32ULU",
    "PASTIS_joint_T30UXV",
}

VAL_FOLDS = [1, 2, 3, 4, 5]
BATCH_SIZE = 16
NUM_WORKERS = 8

results_rows = []


for Target_region in all_tasks_list:
    other_tasks = all_tasks_list - {Target_region}

    for val_fold in VAL_FOLDS:
        val_dataset = PASTIS(
            top_dir=TOP_DIR[Target_region],
            s2_tiles=s2_tiles[Target_region],
            labels=labels[Target_region],
            train_val_key="val",
            val_folds=[val_fold],
        )

        dataloader = torch.utils.data.DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            drop_last=False,
        )

        for source_region in other_tasks:
            model_dir = (
                f"/home/user/results_local/finetuning_results/"
                f"{source_region}/LIANet_valFolds{val_fold}_lr0.0001_batchsize16"
            )

            if not os.path.exists(model_dir):
                continue

            for run_name in sorted(os.listdir(model_dir)):
                ckpt_path = os.path.join(model_dir, run_name, "last.pt")

                if not os.path.exists(ckpt_path):
                    continue

                model = load_model(ckpt_path, source_region)
                model.eval()

                list_of_metrics, _ = multiclass_segmentation_metrics(
                    num_classes=num_classes[Target_region],
                    ignore_index=255,
                )

                metrictracker = MetricCollection(list_of_metrics).to(device)

                with torch.no_grad():
                    for batch in tqdm(
                        dataloader,
                        desc=f"{Target_region} | fold {val_fold} | {source_region} | {run_name}",
                        leave=False,
                    ):
                        x = batch["x_s2"].to(device)
                        y = batch["y_s2"].to(device)
                        label = batch["label"].to(device)
                        delta_days = batch["delta_days"].to(device)
                        target_tile = Target_region.split("_")[-1]
                        region_indx = 0 if target_tile == "T31TFJ" else 1 if target_tile == "T32ULU" else 2 if target_tile == "T31TFM" else 3
                        _, pred = model(
                            delta_days,
                            x,
                            y,
                            torch.tensor([region_indx], device=device),
                        )

                        pred = getattr(pred, "output", pred)

                        if pred.dim() == 3:
                            pred = pred.unsqueeze(1)

                        metrictracker.update(pred, label)

                results = metrictracker.compute()

                row = {
                    "target_region": Target_region,
                    "val_fold": val_fold,
                    "source_region": source_region,
                    "seed_or_run": run_name,
                    "checkpoint_path": ckpt_path,
                }

                for metric_name, metric_value in results.items():
                    row[metric_name] = float(metric_value.detach().cpu())

                results_rows.append(row)

                del model
                del metrictracker
                torch.cuda.empty_cache()

        del dataloader
        del val_dataset
        torch.cuda.empty_cache()


df_results = pd.DataFrame(results_rows)

df_results.to_csv(
    "cr_transfer_results_all_target_regions.csv",
    index=False,
)

df_results

Building val image label pairs: 100%|██████████| 18/18 [00:47<00:00,  2.62s/it]


Found 2736 samples for val


Building val image label pairs: 100%|██████████| 18/18 [00:50<00:00,  2.82s/it]                                            


Found 2610 samples for val


Building val image label pairs: 100%|██████████| 18/18 [00:48<00:00,  2.72s/it]                                            


...

Building val image label pairs: 100%|██████████| 15/15 [00:26<00:00,  1.78s/it]                                          


Found 1665 samples for val


Building val image label pairs: 100%|██████████| 15/15 [00:26<00:00,  1.78s/it]                                            


Found 1845 samples for val


,target_region,val_fold,source_region,seed_or_run,checkpoint_path,accuracy_macro,accuracy_micro,f1_macro,f1_micro,jaccard_macro,jaccard_micro,precision_macro,precision_micro,recall_macro,recall_micro
0,PASTIS_joint_T31TFM,1,PASTIS_joint_T32ULU,2026-05-08_12-42-19,/home/user/results_local/finetuning_results/PA...,0.406962,0.714808,0.359315,0.714808,0.263512,0.556188,0.348022,0.714808,0.406962,0.714808
1,PASTIS_joint_T31TFM,1,PASTIS_joint_T32ULU,2026-05-08_13-28-48,/home/user/results_local/finetuning_results/PA...,0.401282,0.716531,0.355634,0.716531,0.261032,0.558277,0.344434,0.716531,0.401282,0.716531
2,PASTIS_joint_T31TFM,1,PASTIS_joint_T32ULU,2026-05-08_14-16-12,/home/user/results_local/finetuning_results/PA...,0.398334,0.710019,0.352700,0.710019,0.258200,0.550410,0.343265,0.710019,0.398334,0.710019
3,PASTIS_joint_T31TFM,1,PASTIS_joint_T32ULU,2026-05-08_15-03-33,/home/user/results_local/finetuning_results/PA...,0.401586,0.718472,0.355024,0.718472,0.260858,0.560637,0.343015,0.718472,0.401586,0.718472
4,PASTIS_joint_T31TFM,1,PASTIS_joint_T32ULU,2026-05-08_15-50-57,/home/user/results_local/finetuning_results/PA...,0.397532,0.716274,0.355942,0.716274,0.260054,0.557965,0.344863,0.716274,0.397532,0.716274
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,PASTIS_joint_T30UXV,5,PASTIS_joint_T31TFJ,2026-05-09_13-24-19,/home/user/results_local/finetuning_results/PA...,0.189601,0.381220,0.137757,0.381220,0.089370,0.235498,0.201166,0.381220,0.189601,0.381220
296,PASTIS_joint_T30UXV,5,PASTIS_joint_T31TFJ,2026-05-09_14-35-51,/home/user/results_local/finetuning_results/PA...,0.163069,0.346465,0.119693,0.346465,0.076961,0.209530,0.186418,0.346465,0.163069,0.346465
297,PASTIS_joint_T30UXV,5,PASTIS_joint_T31TFJ,2026-05-09_15-47-16,/home/user/results_local/finetuning_results/PA...,0.171923,0.376330,0.133080,0.376330,0.085995,0.231778,0.182961,0.376330,0.171923,0.376330
298,PASTIS_joint_T30UXV,5,PASTIS_joint_T31TFJ,2026-05-09_16-58-42,/home/user/results_local/finetuning_results/PA...,0.182352,0.387289,0.136299,0.387289,0.088141,0.240148,0.172974,0.387289,0.182352,0.387289


In [4]:
metadata_cols = [
    "target_region",
    "val_fold",
    "source_region",
    "seed_or_run",
    "checkpoint_path",
]

metric_cols = [
    col for col in df_results.columns
    if col not in metadata_cols
]

# Average over seeds/runs
df_seed_avg = (
    df_results
    .groupby(["target_region", "val_fold", "source_region"], as_index=False)[metric_cols]
    .mean()
)

# Average over validation folds
df_fold_avg = (
    df_seed_avg
    .groupby(["target_region", "source_region"], as_index=False)[metric_cols]
    .mean()
)

# One result per target region
df_target_avg = (
    df_fold_avg
    .groupby(["target_region"], as_index=False)[metric_cols]
    .mean()
)

# Final result over all target regions
df_final_avg = (
    df_target_avg[metric_cols]
    .mean()
    .to_frame()
    .T
)

df_target_avg.to_csv("cr_transfer_results_avg_per_target_region.csv", index=False)
df_final_avg.to_csv("cr_transfer_results_final_avg_all_regions.csv", index=False)

df_target_avg, df_final_avg

(         target_region  accuracy_macro  accuracy_micro  f1_macro  f1_micro  \
 0  PASTIS_joint_T30UXV        0.275809        0.542107  0.231739  0.542107   
 1  PASTIS_joint_T31TFJ        0.206276        0.519004  0.144921  0.519004   
 2  PASTIS_joint_T31TFM        0.335245        0.602540  0.263280  0.602540   
 3  PASTIS_joint_T32ULU        0.339486        0.546263  0.249242  0.546263   
 
    jaccard_macro  jaccard_micro  precision_macro  precision_micro  \
 0       0.164203       0.379707         0.254348         0.542107   
 1       0.092474       0.351213         0.170490         0.519004   
 2       0.189438       0.439364         0.279651         0.602540   
 3       0.170497       0.381876         0.259468         0.546263   
 
    recall_macro  recall_micro  
 0      0.275809      0.542107  
 1      0.206276      0.519004  
 2      0.335245      0.602540  
 3      0.339486      0.546263  ,
    accuracy_macro  accuracy_micro  f1_macro  f1_micro  jaccard_macro  \
 0        0.

In [5]:
from matplotlib.colors import ListedColormap
colors = [
    (0, 0, 0),
    (0.6823529411764706, 0.7803921568627451, 0.9098039215686274),
    (1.0, 0.4980392156862745, 0.054901960784313725),
    (1.0, 0.7333333333333333, 0.47058823529411764),
    (0.17254901960784313, 0.6274509803921569, 0.17254901960784313),
    (0.596078431372549, 0.8745098039215686, 0.5411764705882353),
    (0.8392156862745098, 0.15294117647058825, 0.1568627450980392),
    (1.0, 0.596078431372549, 0.5882352941176471),
    (0.5803921568627451, 0.403921568627451, 0.7411764705882353),
    (0.7725490196078432, 0.6901960784313725, 0.8352941176470589),
    (0.5490196078431373, 0.33725490196078434, 0.29411764705882354),
    (0.7686274509803922, 0.611764705882353, 0.5803921568627451),
    (0.8901960784313725, 0.4666666666666667, 0.7607843137254902),
    (0.9686274509803922, 0.7137254901960784, 0.8235294117647058),
    (0.4980392156862745, 0.4980392156862745, 0.4980392156862745),
    (0.7803921568627451, 0.7803921568627451, 0.7803921568627451),
    (0.7372549019607844, 0.7411764705882353, 0.13333333333333333),
    (0.8588235294117647, 0.8588235294117647, 0.5529411764705883),
    (0.09019607843137255, 0.7450980392156863, 0.8117647058823529),
    (1, 1, 1),
]
vvmin, vvmax = 0, 19


cmap = ListedColormap(colors)